# Lab 2 — Docker Airflow Pipeline
Run only this notebook for Lab 2. Its prerequisite model preparation is included. Read START_HERE.md for Docker/UI tasks and submission instructions.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys, subprocess, json, time, shutil
uploaded = files.upload()  # Choose Lab2_Docker_Airflow_Pipeline.zip
zip_name = next(name for name in uploaded if name.endswith('.zip'))
base = Path('/content/cartvista-lab2')
base.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    # Reject path traversal; only extract the supplied source package.
    for name in z.namelist():
        if not (base / name).resolve().is_relative_to(base.resolve()):
            raise ValueError('Unsafe ZIP entry')
    z.extractall(base)
PROJECT = base / 'Lab2_Docker_Airflow_Pipeline'
assert (PROJECT / 'requirements.txt').exists()
os.chdir(PROJECT)
print('Project:', PROJECT)


## Install isolated lab environment
The older MLflow/sklearn lab versions need Python 3.11/3.12. The separate environment avoids downgrading Colab's notebook packages. On a future newer hosted Python, this cell obtains a separate Python 3.11 using uv. Setup can take several minutes.


In [ ]:
AUTHOR = 'Sai YourFullName'  # CHANGE THIS BEFORE TRAINING
os.environ['LAB_AUTHOR'] = AUTHOR
VENV = Path('/content/cartvista-lab2-venv')
if sys.version_info[:2] in [(3,11),(3,12)]:
    subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'python', 'install', '3.11'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--seed', '--python', '3.11', str(VENV)], check=True)
PY = str(VENV / 'bin/python')
subprocess.run([PY, '-m', 'pip', 'install', '-r', str(PROJECT/'requirements.txt')], check=True)
def run(*args):
    p = subprocess.Popen([PY, '-u', *args], cwd=PROJECT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines=[]
    for line in p.stdout:
        print(line, end=''); lines.append(line)
    status=p.wait()
    if status: raise subprocess.CalledProcessError(status, args, output=''.join(lines))
    return ''.join(lines)
run('-c', 'import mlflow,sklearn; print(mlflow.__version__,sklearn.__version__)')


## Prerequisite model preparation
This creates this lab’s own data, experiments and champion; do not move another machine’s SQLite registry here.

In [ ]:
run('run_lab1.py')

## Lab 2 Export and batch score
These are real exported-model scoring calls, but **they are not Docker executions**. Build and measure the supplied full/lean Docker images on your laptop to complete that requirement.


In [ ]:
run('export_champion.py')
run('score_batch.py','churn_data.csv','scored.csv')
run('score_batch_lean.py','churn_data.csv','scored_lean.csv')
run('-c', "import pandas as pd,numpy as np; a=pd.read_csv('scored.csv');b=pd.read_csv('scored_lean.csv');assert (a.churn_pred==b.churn_pred).all();assert np.allclose(a.churn_probability,b.churn_probability);print('Full and lean predictions match')")


### Gate logic demonstration
Train two candidates with the same new cycle ID. Compare only those two IDs. The 0.95 run should fail and must not move champion. This cell executes sequentially in Colab; actual Airflow parallel tasks and their screenshots are provided by the Docker route.


In [ ]:
import uuid
cycle = 'colab-'+str(uuid.uuid4())
run('train_tracked.py','--model','logreg','--cycle',cycle,'--output','evidence/colab_lr.json')
run('train_tracked.py','--model','rf','--n-estimators','200','--max-depth','8','--cycle',cycle,'--output','evidence/colab_rf.json')
ids=[json.loads((PROJECT/'evidence'/n).read_text())['run_id'] for n in ['colab_lr.json','colab_rf.json']]
run('pipeline_cli.py','validate','--run-ids',*ids,'--gate','.72','--output',str(PROJECT/'evidence/colab_gate.json'))
winner=json.loads((PROJECT/'evidence/colab_gate.json').read_text())['run_id']
run('pipeline_cli.py','promote','--run-ids',winner)
bad=subprocess.run([PY,'pipeline_cli.py','validate','--run-ids',*ids,'--gate','.95'],cwd=PROJECT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(bad.stdout)
assert bad.returncode != 0, 'Expected deliberately impossible gate to fail'
(PROJECT/'evidence/colab_failed_gate.txt').write_text(bad.stdout)
print('Failure blocked promotion: promote was not called after this gate.')


## Download Lab 2 evidence
Save this executed notebook separately. Complete this lab’s report and any Docker/UI evidence described in START_HERE.md.

In [ ]:
stage=Path('/content/lab2-submission-evidence')
if stage.exists(): shutil.rmtree(stage)
stage.mkdir()
source=PROJECT/'evidence'
if source.exists():
    for f in source.rglob('*'):
        if not f.is_file(): continue
        # Include only this lab's results, plus relevant Colab-specific metrics.
        if f.name.startswith('lab2_') or True and (f.name.startswith('colab_') or 'pipeline' in f.parts or f.name=='gate_failures.log') or False and f.name.startswith('metrics_'):
            d=stage/'evidence'/f.relative_to(source);d.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(f,d)
for name in ['churn_data.csv', 'scored.csv', 'scored_lean.csv']:
    if (PROJECT/name).exists(): shutil.copy2(PROJECT/name,stage/name)
if (PROJECT/'model_export').exists(): shutil.copytree(PROJECT/'model_export',stage/'model_export')
archive=shutil.make_archive('/content/Lab2_My_Run_Evidence','zip',stage)
files.download(archive)
print('Finish the separate Lab 2 report and save this executed notebook.')
